In [ ]:
"""Load Data
Structure:
    1. Imports, Variables, Functions
    2. Load Data
"""

# 1. Imports, Variables, Functions
# imports   
import pandas as pd, numpy as np, os, sys
import anndata as ad
import logging
from typing import *
import pickle
import matplotlib.pyplot as plt
import seaborn as sns
import json
from sklearn.metrics import (
    roc_auc_score,
    roc_curve,
    auc,
    precision_recall_curve,
    average_precision_score,
)
from sklearn.metrics import confusion_matrix, classification_report
import sys
sys.path.append(os.path.join("..", ".."))
from src.utils import utils as ut
from src.utils import viz as vz
from src.utils import io 
logging.basicConfig(level=logging.INFO)

# variables
# run_dir = os.path.join("..","..","outputs","run-25-09-28-05") 
run_dir = os.path.join("..","..","outputs","run-25-09-13-18") 
# run_dir = os.path.join("..","..","outputs","run-25-10-05-01") 
# run_dir = os.path.join("..","..","outputs","run-25-09-17-01") 

embedding_type = "pt"

assert embedding_type in ["ft", "pt", "raw"]
if embedding_type == "ft":
    output_dir = os.path.join(run_dir, "outputs")
elif embedding_type == "pt":
    run_name = run_dir.split("/")[-1]
    output_dir = os.path.join("/aloy/scratch/ddalton/projects/scGPT_playground/outputs/",run_name, "outputs")
elif embedding_type == "raw":
    run_name = run_dir.split("/")[-1]
    output_dir = os.path.join("/aloy/scratch/ddalton/projects/scGPT_playground/outputs/",run_name, "outputs")

if not os.path.exists(output_dir):
    os.makedirs(output_dir)

# functions
def get_counts(df:pd.DataFrame, rel_map:Dict, key_interest:str)->pd.DataFrame:
    df_query = df.copy()

    n_same, n_rel, n_unrel, n_total, n_universe_related, n_total_unique = [], [], [], [], [], []
    n_same_uniq, n_rel_uniq = [], []

    for _, r in df_query.iterrows():
        topk = list(r[key_interest])  # keep duplicates
        doid = r["query_doid"]
        related = rel_map.get(doid, set())

        same = sum(1 for x in topk if x == doid)           # absolute count of the exact disease
        rel  = sum(1 for x in topk if x in related)        # absolute count of related diseases (counts repeats)
        total = len(topk)
        unrel = total - same - rel
        
        n_same.append(same)
        n_same_uniq.append(len(set(topk) & {doid}))
        n_rel.append(rel)
        n_rel_uniq.append(len(set(topk) & related))
        n_unrel.append(unrel)
        n_total.append(total)
        n_universe_related.append(len(related)+1)  # +1 to include self
        n_total_unique.append(len(set(topk)))

    df_query["n_same"]  = n_same
    df_query["n_rel"]   = n_rel
    df_query["n_unrel"] = n_unrel
    df_query["n_total"] = n_total
    df_query["n_universe_related"] = n_universe_related
    df_query["n_total_unique"] = n_total_unique
    df_query["n_same_uniq"] = n_same_uniq
    df_query["n_rel_uniq"] = n_rel_uniq

    df_query["pct_same"]  = df_query["n_same"] / df_query["n_total"] * 100
    df_query["pct_rel"]   = df_query["n_rel"] / df_query["n_total"] * 100
    df_query["pct_unrel"] = df_query["n_unrel"] / df_query["n_total"] * 100

    df_query["hits_same"] = (df_query["n_same"] > 0).astype(int)
    df_query["hits_rel"]  = (df_query["n_rel"] > 0).astype(int)

    # get precision
    # df_query["prec@k"] = (df_query["n_rel"]+df_query["n_same"]) / df_query["n_total"]
    
    df_query["old_prec@k"] = (df_query["n_rel"]+df_query["n_same"]) / df_query["n_total"]
    df_query["old_recall@k"] = (df_query["n_rel"]+df_query["n_same"]) / df_query["n_universe_related"]
    df_query["prec@k"] = (df_query["n_rel_uniq"]+df_query["n_same_uniq"]) / df_query["n_total_unique"]
    df_query["recall@k"] = (df_query["n_rel_uniq"]+df_query["n_same_uniq"]) / df_query["n_universe_related"]

    return df_query

# 2. Load Data
(
    # split,
    predictions_test,
    labels_test,
    results_test,
    all_outputs_test,
    predictions_valid,
    labels_valid,
    results_valid,
    all_outputs_valid,
    predictions_train,
    labels_train,
    results_train,
    all_outputs_train,
    adata_orig,
    id2type,
    train_indices,
    valid_indices,
) = io.load_run_output(run_dir)

# load json

with open(os.path.join(run_dir, "parameters.json"), "r") as f:
    parameters = json.load(f)

for k, v in parameters.items():
    print(f"{k}: {v}")


import importlib
import scanpy as sc

importlib.reload(ut)
importlib.reload(vz)

split_idx = 0

# load all adata
adata_test =  sc.read(
    os.path.join(run_dir, f"adata_test_{split_idx+1}.h5ad"), backed="r"
)
adata_test.obs.reset_index(drop=True, inplace=True)  
adata_valid =  sc.read(
    os.path.join(run_dir, f"adata_valid_{split_idx+1}.h5ad"), backed="r"
)
adata_valid.obs.reset_index(drop=True, inplace=True)  
adata_train =  sc.read(
    os.path.join(run_dir, f"adata_train_{split_idx+1}.h5ad"), backed="r"
)
adata_train.obs.reset_index(drop=True, inplace=True)  




# imports
from tqdm import tqdm
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.multioutput import MultiOutputRegressor
from sklearn.metrics import classification_report
from sklearn.linear_model import SGDRegressor
from sklearn.metrics import mean_squared_error

# variables
remove_related = False
remove_control = True
set_related_pos = False
custom_splits = False
unrel_thr = 0.396
# unrel_thr = 0.0

# functions
def get_mapping_do_sim(do_df_ic:pd.DataFrame, dis_order:List[str], unrel_thr:float=0.0)->Dict:

    do_df_ic_filt = do_df_ic.query("do1 in @dis_order and do2 in @dis_order")
    doid_to_sim = dict()
    for _, r in do_df_ic_filt.iterrows():
        _do1 = r["do1"]
        _do2 = r["do2"]
        _sim = r["lin"]
        doid_to_sim[(_do1, _do2)] = _sim
        doid_to_sim[(_do2, _do1)] = _sim

    pair_sim = dict()
    for d1 in tqdm(dis_order):
        _sim = list()
        for d2 in dis_order:
            if d1 == d2:
                _sim.append(1.0)
            elif (d1 == "Control") or (d2 == "Control"):
                _sim.append(0.0)
            else:
                _sim.append(doid_to_sim[d1, d2] if doid_to_sim[d1, d2]>unrel_thr else 0.0)
        pair_sim[d1] = _sim

    return pair_sim

def get_df_pred(y_prob, y_doids, all_doids):
    results = []
    all_doids = np.array(all_doids)
    for i in range(y_prob.shape[0]):
        y_prob_i = y_prob[i]
        y_doids_i = y_doids[i]  
        
        results.append({
            "query_doid":y_doids_i,
            "top_1":all_doids[np.argsort(y_prob_i)[-1:][::-1]].tolist(),
            "top_3":all_doids[np.argsort(y_prob_i)[-3:][::-1]].tolist(),
            "top_5":all_doids[np.argsort(y_prob_i)[-5:][::-1]].tolist(),
            "top_10":all_doids[np.argsort(y_prob_i)[-10:][::-1]].tolist(),
            "+0.4":all_doids[y_prob_i >= 0.39].tolist(),
            "+0.5":all_doids[y_prob_i >= 0.5].tolist(),
            "+0.6":all_doids[y_prob_i >= 0.6].tolist(),
            "+0.7":all_doids[y_prob_i >= 0.7].tolist(),
            "+0.8":all_doids[y_prob_i >= 0.8].tolist(),
            "+0.9":all_doids[y_prob_i >= 0.9].tolist(),
        })


    return pd.DataFrame(results)

def get_df_performance(dfs_conds:List[Tuple[pd.DataFrame]], rel_map:Dict, n_labels:int)->pd.DataFrame:
    """Get performance metrics aggregated by diseases and by samples
    """
    perf_results = list()
    for _df, _condition in dfs_conds:
        for k in _df.columns[1:]:
            for thr in [0.396, 0.6, 0.8, 0.9]:
                # get counts
                _df_counts = get_counts(_df, rel_map[thr], k)
                _df_counts["random_baseline"] = _df_counts["n_total_unique"]*1/n_labels
                    
                # group by disease
                perf_results.append({
                    "Key": k,
                    "Across": "Diseases",
                    "Condition": _condition,
                    "Hits Same Disease": f"{_df_counts.groupby("query_doid")["hits_same"].mean().mean()*100:.1f}",
                    "Related Threshold": thr,
                    "Precision@K": f"{_df_counts.groupby('query_doid')['prec@k'].mean().mean()*100:.1f}",
                    "Recall@K": f"{_df_counts.groupby('query_doid')['recall@k'].mean().mean()*100:.1f}",    
                    "Size": f'{_df_counts["n_total"].mean():.0f}±{_df_counts["n_total"].std():.0f}' if _df_counts["n_total"].std() > 0 else f'{_df_counts["n_total"].mean():.0f}',
                    "Unique Size": f'{_df_counts["n_total_unique"].mean():.0f}±{_df_counts["n_total_unique"].std():.0f}' if _df_counts["n_total_unique"].std() > 0 else f'{_df_counts["n_total_unique"].mean():.0f}',
                    "Hits Related Disease": f"{_df_counts.groupby("query_doid")["hits_rel"].mean().mean()*100:.1f}",
                    "Random Baseline": f"{_df_counts.groupby('query_doid')['random_baseline'].mean().mean()*100:.1f}",
                })

                # across all samples
                perf_results.append({
                    "Key": k,
                    "Across": "Samples",
                    "Condition": _condition,
                    "Hits Same Disease": f"{_df_counts["hits_same"].mean()*100:.1f}",
                    "Related Threshold": thr,
                    "Precision@K": f"{_df_counts['prec@k'].mean()*100:.1f}",
                    "Recall@K": f"{_df_counts['recall@k'].mean()*100:.1f}",
                    "Size": f'{_df_counts["n_total"].mean():.0f}±{_df_counts["n_total"].std():.0f}' if _df_counts["n_total"].std() > 0 else f'{_df_counts["n_total"].mean():.0f}',
                    "Unique Size": f'{_df_counts["n_total_unique"].mean():.0f}±{_df_counts["n_total_unique"].std():.0f}' if _df_counts["n_total_unique"].std() > 0 else f'{_df_counts["n_total_unique"].mean():.0f}',
                    "Hits Related Disease": f"{_df_counts["hits_rel"].mean()*100:.1f}",
                    "Random Baseline": f"{_df_counts['random_baseline'].mean()*100:.1f}",
                })
    return pd.DataFrame(perf_results)
def custom_train_valid_split(df, random_state=42, remove_control:bool=False)->Tuple[np.ndarray, np.ndarray]:
    """
    For each disease:
      - If multiple datasets exist → split by dataset (1 dataset → valid)
        and also split CONTROL samples belonging to the same datasets.
      - Otherwise → split disease samples by individual rows (1 sample → valid).
    """
    rng = np.random.default_rng(random_state)
    train_idx, valid_idx = [], []

    if not remove_control:
        df_controls = df[df["doid_id"] == "Control"]

    valid_datasets = set()
    train_datasets = set()
    # Loop through diseases
    for doid in df["doid_id"].unique():
        if doid == "Control":
            continue  # df_controls handled when splitting real diseases
        
        df_do = df[df["doid_id"] == doid]
        datasets = df_do["dataset"].unique()

        n_overlap_valid = len(valid_datasets & set(datasets))
        if (n_overlap_valid>0) & (len(datasets)-n_overlap_valid)>0:   # if dataset already placed in valid set

            # place existing dataset in valid for this disease
            valid_idx.extend(df_do[df_do["dataset"].isin(valid_datasets)].index)
            train_idx.extend(df_do[~df_do["dataset"].isin(valid_datasets)].index)

        elif len(datasets)-n_overlap_valid > 1:
            rng.shuffle(list(datasets))
            new_valid_datasets = {d for d in datasets if d not in valid_datasets}  # datasets not already in valid
            new_valid_datasets = {list(new_valid_datasets)[0]}  # take one new dataset for valid
            new_train_datasets = set(datasets) - new_valid_datasets - valid_datasets

            # disease samples
            valid_idx.extend(df_do[df_do["dataset"].isin(new_valid_datasets)].index)
            train_idx.extend(df_do[df_do["dataset"].isin(new_train_datasets)].index)

            valid_datasets.update(new_valid_datasets)
            train_datasets.update(new_train_datasets)

            if not remove_control:
                # matching CONTROL samples for those datasets
                ctrl_valid = df_controls[df_controls["dataset"].isin(new_valid_datasets)].index
                ctrl_train = df_controls[df_controls["dataset"].isin(new_train_datasets)].index

                valid_idx.extend(ctrl_valid)
                train_idx.extend(ctrl_train)
        else:
            idx = df_do.index.to_numpy()
            rng.shuffle(idx)
            n_valid_ds = max(1, len(idx) // 6)
            valid_idx.append(idx[n_valid_ds-1])
            train_idx.extend(idx[n_valid_ds:])

            # controls for this dataset (if available)
            ds = df_do["dataset"].iloc[0]

            if not remove_control:            
                ctrl_ds = df_controls[df_controls["dataset"] == ds].index.to_numpy()

                if len(ctrl_ds) > 0:
                    rng.shuffle(ctrl_ds)
                    n_valid_ds = max(1, len(ctrl_ds) // 6)
                    valid_idx.append(ctrl_ds[n_valid_ds-1])
                    train_idx.extend(ctrl_ds[n_valid_ds:])

    return np.array(train_idx), np.array(valid_idx)

def get_multilabel_y(df:pd.DataFrame, all_doids:List[str], rel_map:Dict, set_related_pos:bool, thr_related:int=0.396)->np.ndarray:
    y_train = np.zeros((len(df), len(all_doids)), dtype=int)
    doids = df["doid_id"].to_list()
    for i in range(y_train.shape[0]):
        doid = doids[i]
        if set_related_pos:
            related = rel_map[thr_related].get(doid, set())
            # add self
            related.add(doid)
            idxs = [all_doids.index(d) for d in related]
        else:
            idxs = [all_doids.index(doid)]
        
        # set positive labels
        y_train[i, idxs] = 1
    return y_train

# prepare data
all_doids = sorted(adata_train.obs["doid_id"].unique())

df_train = adata_train.obs.copy()
df_valid = adata_valid.obs.copy()
df_test = adata_test.obs.copy()

e_train = embeddings_train
e_valid = embeddings_valid
e_test = embeddings_test

if remove_control:
    all_doids.remove("Control")

    mask_train = df_train["doid_id"] != "Control"
    mask_valid = df_valid["doid_id"] != "Control"
    mask_test = df_test["doid_id"] != "Control"

    df_train = df_train[mask_train].reset_index(drop=True)
    df_valid = df_valid[mask_valid].reset_index(drop=True)
    df_test = df_test[mask_test].reset_index(drop=True)

    e_train = e_train[mask_train.values]
    e_valid = e_valid[mask_valid.values]
    e_test = e_test[mask_test.values]


if custom_splits:
    df_train_all = pd.concat([df_train.reset_index(drop=True), df_valid.reset_index(drop=True)], axis=0)
    df_train_all = df_train_all.reset_index(drop=True)

    # RE-SPLIT TRAIN/VALIDATION SET
    train_idx, valid_idx = custom_train_valid_split(df_train_all, remove_control=remove_control)
    df_train = df_train_all.iloc[train_idx].reset_index(drop=True)
    df_valid = df_train_all.iloc[valid_idx].reset_index(drop=True)

    e_train_all = np.concatenate([e_train, e_valid], axis=0)
    e_train = e_train_all[train_idx]
    e_valid = e_train_all[valid_idx]

dis_order = df_train["doid_id"].unique()
print(f"Total dis_ordereases in training set: {len(dis_order)}")

# Encode labels
pair_sim = get_mapping_do_sim(do_df_ic, all_doids, unrel_thr=unrel_thr)
y_train = np.array([pair_sim[x] for x in df_train["doid_id"].values])
y_valid = np.array([pair_sim[x] for x in df_valid["doid_id"].values])
y_test = np.array([pair_sim[x] for x in df_test["doid_id"].values])

# multilabels
y_test_multilab = get_multilabel_y(df_test, all_doids, rel_map, set_related_pos)
y_train_multilab = get_multilabel_y(df_train, all_doids, rel_map, set_related_pos)
y_valid_multilab = get_multilabel_y(df_valid, all_doids, rel_map, set_related_pos)

print(f"Nº of train samples: {df_train.shape[0]}\tNº of diseases: {len(df_train['doid_id'].unique())}")
print(f"Nº of validation samples: {df_valid.shape[0]}\tNº of diseases: {len(df_valid['doid_id'].unique())}")
print(f"Nº of test samples: {df_test.shape[0]}\tNº of diseases: {len(df_test['doid_id'].unique())}")
